# SKB-Q / BASC-20 — code-only prospective validation

This notebook requires the original, separately supplied frozen manifest. It does not contain allocation maps, model checkpoints, proprietary research data, or outcome logs. Set `SKBQ_BASC_MANIFEST` to the manifest JSON path. The original experimental result is unchanged; the code-only release is not a new independent GPU evaluation.


In [ ]:
import sys,subprocess,importlib.metadata as meta
wanted={'transformers':'4.49.0','datasets':'3.5.1','huggingface-hub':'0.36.2','safetensors':'0.5.3'}
missing=[]
for pkg,v in wanted.items():
 try: installed=meta.version(pkg)
 except meta.PackageNotFoundError: installed=''
 if installed!=v:missing.append(f'{pkg}=={v}')
if missing:
 print('Installing pinned dependencies:',missing,flush=True)
 subprocess.check_call([sys.executable,'-m','pip','install','-q','--disable-pip-version-check',*missing])
print('Pinned dependencies ready; restart notebook if you had already imported transformers/datasets in this kernel.')

In [ ]:
import io,os,json,hashlib,zipfile,base64,math,shutil,time
from pathlib import Path
import numpy as np
import torch,torch.nn as nn,torch.nn.functional as F
from transformers import AutoTokenizer,AutoModelForCausalLM
from datasets import load_dataset
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before running.'
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False

def canon(x):return json.dumps(x,sort_keys=True,ensure_ascii=False,separators=(',',':'),allow_nan=False).encode('utf-8')
def sha(x):return hashlib.sha256(x).hexdigest()
OUT=Path('/kaggle/working/SKBQ_BASC20_VALIDATION_FULL');OUT.mkdir(parents=True,exist_ok=True)
def save(name,obj):
 p=OUT/name;tmp=p.with_suffix(p.suffix+'.tmp')
 tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8');tmp.replace(p)

# Code-only distribution: frozen allocation maps are external research data.
# Supply an authorized manifest separately; this notebook publishes no maps or outcomes.
MANIFEST_PATH=Path(os.environ.get('SKBQ_BASC_MANIFEST', '/kaggle/input/skbq-private/BASC20_FROZEN_MANIFEST.json'))
if not MANIFEST_PATH.exists():
 raise FileNotFoundError('Supply the original frozen BASC20 manifest via SKBQ_BASC_MANIFEST')
lock=json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
core={k:v for k,v in lock.items() if k!='canonical_self_sha256'}
EXPECTED_FROZEN_MANIFEST_SHA256=lock['canonical_self_sha256']
assert sha(canon(core))==EXPECTED_FROZEN_MANIFEST_SHA256,'Manifest checksum mismatch'
assert lock['state']=='FROZEN_MAPS_TRAINING_ONLY_TARGET_UNOBSERVED'
assert 'validation split' in lock['data']['target'].lower()
PIN=lock['model_revision'];MODEL_ID=lock['model_id'];BUDGET=4.1997197809196285
assert (PIN,MODEL_ID)==('8faed761d45a263340a0528343f099c05c9a4323','Qwen/Qwen2.5-1.5B')
MAPS=lock['chosen_maps']
ORDER=['BASC20','Ridge20','Lasso20','Ridge30','Lasso30','BestDev20','BestDev30','WeightNormTopK','RTNErrorTopK']
assert set(ORDER)==set(MAPS)
for p,m in MAPS.items():assert len(m)==14 and len(set(m))==14 and sorted(m)==m and all(0<=int(v)<28 for v in m)
assert len({tuple(m) for m in MAPS.values()})==len(MAPS)
plan={'study_id':'SKBQ-BASC20-VALIDATION-v1','model_id':MODEL_ID,'model_revision':PIN,
 'frozen_manifest_canonical_sha256':EXPECTED_FROZEN_MANIFEST_SHA256,'policy_order':ORDER,'maps':MAPS,
 'target_split':'validation','source':'Salesforce/wikitext','config':'wikitext-2-raw-v1',
 'dataset_revision':'f776294184f13b8ff2337b3841cf9269a6216d1e','n_blocks':64,
 'block_tokens':1024,'predictions_per_block':1023,
 'nominal_weighted_bit_budget':BUDGET,'metric':'mean per-block next-token cross entropy NLL; exp for perplexity',
 'a_priori_comparisons':['BASC20-Ridge20','BASC20-Lasso20','BASC20-Ridge30','BASC20-Lasso30'],
 'secondary_comparisons':['BASC20-BestDev20','BASC20-BestDev30','BASC20-WeightNormTopK','BASC20-RTNErrorTopK'],
 'uncertainty':'5000 paired iid-block descriptive bootstrap draws, seed 20261019; adjacent blocks may share a document',
 'no_refitting_on_validation':True}
plan_sha=sha(canon(plan))
save('FROZEN_MANIFEST_COPY.json',lock)
save('PLAN_BEFORE_OUTCOMES.json',dict(plan,canonical_plan_sha256=plan_sha))
print('Frozen selector map:', MAPS['BASC20'])
print('Frozen hash:',EXPECTED_FROZEN_MANIFEST_SHA256,'plan hash:',plan_sha)

In [ ]:
# Pinned architecture and weights, inherited verbatim in semantics from the audited Stage-B protocol.
tok=AutoTokenizer.from_pretrained(MODEL_ID,revision=PIN,trust_remote_code=False)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=PIN,torch_dtype=torch.float16,
 attn_implementation='sdpa',trust_remote_code=False,low_cpu_mem_usage=True).eval().to('cuda')
model.config.use_cache=False
cfg=model.config
assert cfg.model_type=='qwen2' and cfg.num_hidden_layers==28 and cfg.hidden_size==1536
assert cfg.num_attention_heads==12 and cfg.num_key_value_heads==2
if model.lm_head.weight.data_ptr()==model.get_input_embeddings().weight.data_ptr():
 model.lm_head.weight=nn.Parameter(model.lm_head.weight.detach().clone(),requires_grad=False)
assert model.lm_head.weight.data_ptr()!=model.get_input_embeddings().weight.data_ptr()
emb_sha=sha(model.get_input_embeddings().weight.detach().cpu().numpy().tobytes())
EXPECTED_EMBEDDING_SHA='fdb9e308aa4aab913119197abfbabf3bdee64178fc91348f6a759a2e53da07b8'
assert emb_sha==EXPECTED_EMBEDDING_SHA,'Model weights/revision differs from prior Stage B, STOP'
modules={n:m for n,m in model.named_modules() if isinstance(m,nn.Linear)}
expected={f'model.layers.{i}.self_attn.{p}' for i in range(28) for p in ('q_proj','k_proj','v_proj','o_proj')}
expected|={f'model.layers.{i}.mlp.{p}' for i in range(28) for p in ('gate_proj','up_proj','down_proj')}
expected.add('lm_head')
assert set(modules)==expected,'Model named Linear modules differ from prior protocol'
WEIGHT_COUNT=sum(m.weight.numel() for m in modules.values())
GROUP_WEIGHT=sum(modules[f'model.layers.0.self_attn.{p}'].weight.numel() for p in ('q_proj','k_proj','v_proj','o_proj'))
assert (WEIGHT_COUNT,GROUP_WEIGHT)==(1543569408,5505024)
assert abs(4+4*14*GROUP_WEIGHT/WEIGHT_COUNT-BUDGET)<1e-12
master={n:m.weight.detach().cpu().clone() for n,m in modules.items()}
save('MODEL_AND_BUDGET_LOCK.json',{'model_revision':PIN,'torch':str(torch.__version__),
 'transformers':__import__('transformers').__version__,'embedding_sha256':emb_sha,
 'linear_params':WEIGHT_COUNT,'group_params':GROUP_WEIGHT,'frozen_manifest_sha256':EXPECTED_FROZEN_MANIFEST_SHA256,
 'plan_sha256':plan_sha})
print('Locked model and exact counted budget')

In [ ]:
# Lock target tokens BEFORE computing any target losses.
# Use identical tokenization logic to the previous Stage-B study; only split changes to validation.
ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',revision='f776294184f13b8ff2337b3841cf9269a6216d1e',split='validation')
texts=[str(r['text']).strip() for r in ds if r['text'] and str(r['text']).strip()]
ids=tok('\n\n'.join(texts),add_special_tokens=False)['input_ids']
COUNT=64;L=1024
assert len(ids)>=COUNT*L,'Not enough validation tokens for the locked 64-block protocol.'
blocks=np.asarray(ids[:COUNT*L],dtype='<i4').reshape(COUNT,L)
target_sha=sha(blocks.tobytes())
# Distinct from the already evaluated WikiText-2 test token hash.
assert target_sha!='ebf30d98a1f52b3ae8d96136565115ae8965e2170dc38dcdc7723e76d46cc163'
save('VALIDATION_TOKENS_LOCK_BEFORE_OUTCOMES.json',{'source':'Salesforce/wikitext','split':'validation',
 'revision':'f776294184f13b8ff2337b3841cf9269a6216d1e','token_sha256':target_sha,
 'n_blocks':COUNT,'length':L,'predictions':COUNT*(L-1),
 'frozen_manifest_sha256':EXPECTED_FROZEN_MANIFEST_SHA256,'plan_sha256':plan_sha})
print('Validation token hash locked:',target_sha,'tokens',COUNT*L)

In [ ]:
def rtn_row_chunk(w,bits):
 qmax=(1<<(bits-1))-1
 v=w.to(device='cuda',dtype=torch.float32)
 scale=v.abs().amax(dim=1,keepdim=True)/qmax
 scale=torch.where(scale>0,scale,torch.ones_like(scale))
 return ((v/scale).round().clamp(-qmax,qmax)*scale).to(torch.float16)

def map_bits(policy):
 selected=MAPS[policy]
 bits={n:4 for n in modules}
 for layer in selected:
  for p in ('q_proj','k_proj','v_proj','o_proj'):
   bits[f'model.layers.{layer}.self_attn.{p}']=8
 avg=sum(modules[n].weight.numel()*b for n,b in bits.items())/WEIGHT_COUNT
 assert abs(avg-BUDGET)<1e-12
 return bits,avg

def install_bits(mapping):
 with torch.no_grad():
  for n,m in modules.items():
   original=master[n]
   for st in range(0,original.shape[0],4096):
    en=min(st+4096,original.shape[0]);src=original[st:en]
    m.weight.data[st:en].copy_(rtn_row_chunk(src,mapping[n]))

allresults=[]
for i,policy in enumerate(ORDER,1):
 mapping,avgbit=map_bits(policy)
 map_sha=sha(canon(mapping))
 fp=sha(canon({'plan_sha256':plan_sha,'frozen_manifest_sha256':EXPECTED_FROZEN_MANIFEST_SHA256,
 'model_revision':PIN,'policy':policy,'map_sha256':map_sha,
 'token_sha256':target_sha,'avgbit':avgbit}))
 filename=f'checkpoint_{policy}.json';checkpoint=OUT/filename
 vals=[]
 if checkpoint.exists():
  prev=json.loads(checkpoint.read_text());assert prev['fingerprint']==fp,'Incompatible checkpoint: STOP'
  vals=list(map(float,prev['block_mean_nll']));assert len(vals)<=COUNT
 print(f'[{i}/{len(ORDER)}] {policy} — resume {len(vals)}/64',flush=True)
 install_bits(mapping)
 start=time.monotonic()
 for bi in range(len(vals),COUNT):
  x=torch.as_tensor(blocks[bi],dtype=torch.long,device='cuda').unsqueeze(0)
  with torch.inference_mode():
   logits=model(input_ids=x,use_cache=False).logits[:,:-1,:].float().contiguous()
   loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),x[:,1:].reshape(-1))
   value=float(loss.item())
  assert math.isfinite(value)
  vals.append(value)
  del x,logits,loss
  if (bi+1)%4==0 or bi==COUNT-1:
   save(filename,{'policy':policy,'selected_layers':MAPS[policy],
   'avgbit_linear':avgbit,'block_mean_nll':vals,'blocks_complete':len(vals),
   'frozen_manifest_sha256':EXPECTED_FROZEN_MANIFEST_SHA256,
   'plan_sha256':plan_sha,'target_token_sha256':target_sha,
   'allocation_sha256':map_sha,'fingerprint':fp})
   print(f'  {bi+1}/64, running mean NLL {float(np.mean(vals)):.6f}',flush=True)
 allresults.append({'policy':policy,'avgbit_linear':avgbit,'blocks':COUNT,
 'predicted_tokens':COUNT*(L-1),'mean_nll':float(np.mean(vals)),
 'ppl':float(math.exp(float(np.mean(vals)))),'allocation_sha256':map_sha,
 'incremental_eval_wall_seconds_this_run':float(time.monotonic()-start)})
 save('POLICY_RESULTS.json',allresults)
 torch.cuda.empty_cache()
 # Incremental backup archive for short-lived Kaggle sessions.
 shutil.make_archive(str(OUT),'zip',root_dir=OUT)
assert len(allresults)==len(ORDER)
assert sha(model.get_input_embeddings().weight.detach().cpu().numpy().tobytes())==emb_sha
print('All nine frozen policies evaluated')

In [ ]:
# Frozen comparison plan: no retuning after outcomes, including if BASC loses.
losses={}
for name in ORDER:
 ck=json.loads((OUT/f'checkpoint_{name}.json').read_text())
 assert ck['blocks_complete']==64 and ck['plan_sha256']==plan_sha
 assert ck['target_token_sha256']==target_sha
 assert ck['frozen_manifest_sha256']==EXPECTED_FROZEN_MANIFEST_SHA256
 arr=np.asarray(ck['block_mean_nll'],dtype=np.float64)
 assert arr.shape==(64,) and np.isfinite(arr).all()
 assert abs(float(arr.mean())-next(r['mean_nll'] for r in allresults if r['policy']==name))<1e-12
 losses[name]=arr
rng=np.random.default_rng(20261019)
resampled=rng.integers(0,64,size=(5000,64))
comparisons={}
for comparator in ORDER[1:]:
 diff=losses['BASC20']-losses[comparator]
 boot=diff[resampled].mean(axis=1)
 comparisons[comparator]={'BASC_minus_comparator_mean_NLL':float(diff.mean()),
 'bootstrap_iid_block_CI95_descriptive':[float(np.quantile(boot,.025)),float(np.quantile(boot,.975))],
 'BASC_lower_block_count':int((diff<0).sum()),
 'BASC_higher_block_count':int((diff>0).sum()),
 'relative_PPL_reduction_if_positive':float(1-np.exp(diff.mean()))}
summary={'study_id':plan['study_id'],'target_split':'validation',
 'frozen_manifest_sha256':EXPECTED_FROZEN_MANIFEST_SHA256,'plan_sha256':plan_sha,
 'target_token_sha256':target_sha,'paired_comparisons':comparisons,
 'primary_comparators':['Ridge20','Lasso20','Ridge30','Lasso30'],
 'limitations':['One data split/model/selection instance; descriptive bootstrap only',
 'Methods informed by older test-set experiments despite new validation labels',
 'No independent multiple-run selection evidence or quantizer transfer',
 '20 vs 30 labels count is protocol cost, not measured Kaggle runtime savings']}
save('BASC20_PAIRED_COMPARISONS.json',summary)
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Upload to researcher:',archive)
for method,c in comparisons.items():print('BASC20 -',method,round(c['BASC_minus_comparator_mean_NLL'],7),c['bootstrap_iid_block_CI95_descriptive'])

## Research note
Upload frozen manifest privately when running. This code-only publication does not authorize sharing experiment outputs or private data.
